# Week 3: Experiment pipeline

[7 total sections]
| |  |
|---|---|
| 1. Setup | common |
| 2. Shared code | common |
| 3. Augmentation | **Data** |
| 4. Training and fine-tuning | **Model** |
| 5. Evaluation and logging | **Eval** |
| 6. Grad-CAM | **XAI** |
| 7. Combining | common |

- Only train and validation data are used.

## 1. Setup

(run at the start of every session)

In [ ]:
# 1. Setup
!pip install grad-cam --quiet

from google.colab import drive
drive.mount('/content/drive')

import os, shutil, json, random, time, csv
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

PROJECT = "/content/drive/MyDrive/fruit_project"
DATA = "/content/dataset"
if not os.path.isdir(DATA):
    shutil.unpack_archive(f"{PROJECT}/dataset.zip", DATA)     # takes a minute or two

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)
assert device.type == "cuda", "Runtime -> Change runtime type -> GPU, then rerun everything"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)

split = json.load(open(f"{PROJECT}/split.json"))
probe = datasets.ImageFolder(DATA)
assert probe.class_to_idx == split["class_to_idx"], "class order changed"
assert len(probe) == sum(len(split[k]) for k in ["train", "val", "test"]), "image count mismatch"

class_names = probe.classes
num_classes = len(class_names)
HEALTHY = probe.class_to_idx["Healthy"]
for sub in ["checkpoints", "results"]:
    os.makedirs(f"{PROJECT}/{sub}", exist_ok=True)
print("OK:", probe.class_to_idx, len(probe), "images")

## 2. Shared code

In [ ]:
# 2. Shared pieces
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]     # ImageNet statistics
EVAL_TFM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

def make_datasets(train_tfm):
    train_full = datasets.ImageFolder(DATA, transform=train_tfm)   # training images may be augmented
    eval_full = datasets.ImageFolder(DATA, transform=EVAL_TFM)     # validation images never augmented
    return Subset(train_full, split["train"]), Subset(eval_full, split["val"])

def make_loaders(train_ds, val_ds, bs=32):
    return (DataLoader(train_ds, bs, shuffle=True, num_workers=2),
            DataLoader(val_ds, bs, shuffle=False, num_workers=2))

# Class weights come from the train split only. Rarer classes count for more in the loss.
train_targets = np.array(probe.targets)[split["train"]]
counts = np.bincount(train_targets, minlength=num_classes)
class_weights = torch.tensor(len(train_targets) / (num_classes * counts), dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
print("training images per class:", dict(zip(class_names, counts.tolist())))

def build_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)   # pretrained on ImageNet
    for p in model.parameters():
        p.requires_grad = False                                        # freeze everything
    model.fc = nn.Linear(model.fc.in_features, num_classes)            # new trainable head
    return model.to(device)

def load_model(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model = build_model(len(ckpt["class_names"]))
    model.load_state_dict(ckpt["model_state_dict"])
    return model.eval()

def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over the data. Trains if an optimizer is given, otherwise only evaluates."""
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

def predict_all(model, loader):
    model.eval()
    ys, ps, probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            p = torch.softmax(model(x.to(device)), dim=1).cpu()
            probs.append(p.numpy()); ps.append(p.argmax(1).numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(probs)

## 3. Augmentation

Augmentation randomly alters training images each time the model sees them, so it cannot memorise exact photos.

Note:
1. Rot and lesions are colour cues so color changes can erase the signal for them. So color jitter must be kept low.
2. Cropping can cut the defect out, so the model would see a healthy looking apple but the label would be "Black Rot". So the crop scale must be kept high so very little is cropped out.

In [ ]:
# 3. Augmentation settings
PIL_STEPS = {
    "none": [
        transforms.Resize((224, 224)),
    ],
    "light": [
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
    ],
    "aggressive": [
        transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(25),
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    ],
}
TAIL = [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]

def get_train_tfm(name):
    return transforms.Compose(PIL_STEPS[name] + TAIL)

def sample_train_image(class_name):
    """A random training image of the given class."""
    cls = probe.class_to_idx[class_name]
    idxs = [i for i in split["train"] if probe.targets[i] == cls]
    return probe.samples[random.choice(idxs)][0]

def preview_augmentation(path, aug_name, class_name, n=6):
    img = Image.open(path).convert("RGB")
    t = transforms.Compose(PIL_STEPS[aug_name])            #so the result displays normally
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
    for ax in axes:
        ax.imshow(t(img)); ax.axis("off")
    fig.suptitle(f"{class_name} | augmentation: {aug_name}")
    plt.tight_layout()
    plt.savefig(f"{PROJECT}/results/aug_{aug_name}_{class_name.replace(' ', '_')}.png", dpi=120)
    plt.show()

# Look at all three settings on one photo of each defect class
for cname in ["Black Rot", "Black Pox", "Powdery Mildew"]:
    path = sample_train_image(cname)
    for aug_name in PIL_STEPS:
        preview_augmentation(path, aug_name, cname)

## 4. Training and fine-tuning

- `fit` trains for N epochs and saves the best epoch (lowest validation loss).
- `set_finetune_mode` picks how much of the pretrained network may change: `frozen` (new head only), `last_block` (head + the last convolutional block), `full` (everything).
- `make_optimizer` gives the new head a normal learning rate and the pretrained parts a much smaller one. Large steps can destroy the the highly optimized, general-purpose features that a CNN backbone has already learned

In [ ]:
# 4. Training and fine-tuning
def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def set_finetune_mode(model, mode):
    for p in model.parameters():
        p.requires_grad = False
    parts = {"frozen": [model.fc],
             "last_block": [model.layer4, model.fc],
             "full": [model]}[mode]
    for part in parts:
        for p in part.parameters():
            p.requires_grad = True

def make_optimizer(model, mode):
    head = list(model.fc.parameters())
    groups = [{"params": head, "lr": 1e-3}]                              # the new head always learns at 1e-3
    if mode == "last_block":
        groups.append({"params": list(model.layer4.parameters()), "lr": 1e-4})
    elif mode == "full":
        backbone = [p for n, p in model.named_parameters() if not n.startswith("fc.")]
        groups.append({"params": backbone, "lr": 1e-5})
    return torch.optim.Adam(groups)

def fit(model, train_loader, val_loader, criterion, optimizer, epochs, ckpt_path, class_names, config=None):
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss, best_epoch = float("inf"), 0
    t_start = time.time()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader, criterion)
        for k, v in zip(history, (tl, ta, vl, va)):
            history[k].append(v)
        note = ""
        if vl < best_val_loss:
            best_val_loss, best_epoch = vl, epoch
            torch.save({"model_state_dict": model.state_dict(), "class_names": class_names,
                        "epoch": epoch, "val_loss": vl, "val_acc": va, "config": config}, ckpt_path)
            note = "  <- best so far, saved"
        print(f"epoch {epoch:2d} | train loss {tl:.3f} acc {ta:.3f} | "
              f"val loss {vl:.3f} acc {va:.3f} | {time.time() - t0:.0f}s{note}")
    return history, best_epoch, (time.time() - t_start) / epochs

In [ ]:
# how many weights can each mode change?
m = build_model(num_classes)
for mode in ["frozen", "last_block", "full"]:
    set_finetune_mode(m, mode)
    print(f"{mode:11s} trainable parameters: {count_trainable(m):,}")
# Expect: frozen = 2,052 (the new 512x4 head plus 4 biases); full = about 11.2 million; last_block in between.
del m

## 5. Evaluation and logging (using validation set)


- **Miss rate/FNR:** of the diseased fruit, how many were called Healthy?
- **Macro-F1:** F1 computed per class, then averaged equally. Rare classes count as much as common ones.
- **Most-confused pair:** the off-diagonal cell with the most images.

(`evaluate_run` works on any saved model, and every experiment is saved as row in `experiments.csv`)

In [ ]:
# 5. Evaluation and logging
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

def plot_history(h, save_path=None):
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(h["train_loss"], label="train"); ax[0].plot(h["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h["train_acc"], label="train"); ax[1].plot(h["val_acc"], label="val")
    ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

def evaluate_run(model, loader, class_names, save_prefix):
    y, p, _ = predict_all(model, loader)
    print(classification_report(y, p, target_names=class_names, digits=3))

    cm = confusion_matrix(y, p, labels=list(range(len(class_names))))
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax[0], cmap="Blues", xticks_rotation=45, colorbar=False)
    ax[0].set_title("Image counts")
    ConfusionMatrixDisplay(cm / cm.sum(axis=1, keepdims=True), display_labels=class_names).plot(
        ax=ax[1], cmap="Blues", xticks_rotation=45, colorbar=False, values_format=".2f")
    ax[1].set_title("Share of each true class")
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_confusion.png", dpi=150)
    plt.show()

    recall = np.diag(cm) / cm.sum(axis=1)
    off = cm.copy(); np.fill_diagonal(off, 0)
    i, j = np.unravel_index(off.argmax(), off.shape)
    diseased, healthy = (y != HEALTHY), (y == HEALTHY)
    metrics = {
        "val_acc": float((y == p).mean()),
        "val_macro_f1": float(f1_score(y, p, average="macro")),
        "miss_rate": float(((p == HEALTHY) & diseased).sum() / diseased.sum()),
        "false_alarm_rate": float(((p != HEALTHY) & healthy).sum() / healthy.sum()),
        "weakest_class": class_names[int(recall.argmin())],
        "weakest_recall": float(recall.min()),
        "most_confused": f"{class_names[i]} -> {class_names[j]} ({off[i, j]} images)",
    }
    print(f"accuracy {metrics['val_acc']:.3f} | macro-F1 {metrics['val_macro_f1']:.3f}")
    print(f"miss rate (diseased called Healthy): {metrics['miss_rate']:.3f} | "
          f"false alarms (Healthy called diseased): {metrics['false_alarm_rate']:.3f}")
    print(f"weakest class: {metrics['weakest_class']} (recall {metrics['weakest_recall']:.3f}) | "
          f"most confused: {metrics['most_confused']}")
    return metrics

LOG_COLUMNS = ["run", "augmentation", "finetune_mode", "epochs", "best_epoch", "seed", "sec_per_epoch",
               "val_acc", "val_macro_f1", "miss_rate", "false_alarm_rate",
               "weakest_class", "weakest_recall", "most_confused", "notes"]

def log_result(run, aug, mode, epochs, best_epoch, seed, sec_per_epoch, metrics, notes=""):
    path = f"{PROJECT}/results/experiments.csv"
    new = not os.path.exists(path)
    row = {"run": run, "augmentation": aug, "finetune_mode": mode, "epochs": epochs,
           "best_epoch": best_epoch, "seed": seed, "sec_per_epoch": round(sec_per_epoch, 1), "notes": notes}
    row.update({k: (round(v, 4) if isinstance(v, float) else v) for k, v in metrics.items()})
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLUMNS)
        if new:
            w.writeheader()
        w.writerow(row)

## 6. Grad-CAM (XAI)

Grad-CAM shows which parts of a photo pushed the model toward its answer. Warm colours indicate the regions used to come to a decision. The model may arrive at a decision based on irrelevant things like the background, a leaf, etc.


In [ ]:
# 6. Grad-CAM
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


def misclassified_indices(model, dataset, bs=64):
    """Positions in dataset that the model gets wrong."""
    loader = DataLoader(dataset, batch_size=bs, shuffle=False, num_workers=2)
    y, p, _ = predict_all(model, loader)
    return np.where(y != p)[0].tolist()

def gradcam_grid(model, dataset, class_names, indices=None, n=6, save_path=None, seed=0):
    # Grad-CAM needs gradients to flow through the whole network. With a frozen backbone they would not,
    # so we switch them back on for explainability.
    for p in model.parameters():
        p.requires_grad = True
    model.eval()
    cam = GradCAM(model=model, target_layers=[model.layer4[-1]])      # last conv block of ResNet18

    if indices is None:
        indices = random.Random(seed).sample(range(len(dataset)), n)
    indices = indices[:n]
    if len(indices) == 0:
        print("nothing to show")
        return

    fig, axes = plt.subplots(2, len(indices), figsize=(3 * len(indices), 6.6), squeeze=False)
    for col, i in enumerate(indices):
        x, y = dataset[i]
        inp = x.unsqueeze(0).to(device)
        probs = torch.softmax(model(inp), dim=1)[0].detach()
        pred = int(probs.argmax())
        target_category = int(probs.argmax())
        targets = [ClassifierOutputTarget(target_category)]
        heat = cam(input_tensor=inp, targets=targets)[0]
                              # explains the predicted class
        rgb = (x.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1).astype(np.float32)   # undo normalisation

        axes[0][col].imshow(rgb); axes[0][col].axis("off")
        axes[0][col].set_title(f"true: {class_names[y]}", fontsize=9)
        axes[1][col].imshow(show_cam_on_image(rgb, heat, use_rgb=True)); axes[1][col].axis("off")
        axes[1][col].set_title(f"pred: {class_names[pred]} ({probs[pred]:.0%})", fontsize=9,
                               color="green" if pred == y else "red")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()

## 7. Combining

- `run_experiment` builds the data with the chosen augmentation, trains with the chosen fine-tuning mode, saves the best checkpoint, evaluates it, logs a row to `experiments.csv`, and draws Grad-CAM for random and for misclassified validation images.
- `seed` controls the random shuffling and augmentation. Repeating a run with a different seed shows how much the numbers change, which indicates how big a difference has to be before it means anything.

In [ ]:
# 7. run_experiment
def run_experiment(name, aug="none", mode="frozen", epochs=10, bs=32, seed=42,
                   gradcam=True, log=True, overwrite=False, notes=""):
    ckpt = f"{PROJECT}/checkpoints/{name}.pt"
    assert overwrite or not os.path.exists(ckpt), f"'{name}' already exists. Pick a new name."
    set_seed(seed)

    train_ds, val_ds = make_datasets(get_train_tfm(aug))
    train_loader, val_loader = make_loaders(train_ds, val_ds, bs)

    model = build_model(num_classes)
    set_finetune_mode(model, mode)
    optimizer = make_optimizer(model, mode)
    print(f"=== {name} | aug={aug} | mode={mode} | epochs={epochs} | seed={seed} "
          f"| trainable weights={count_trainable(model):,}")

    config = {"aug": aug, "mode": mode, "epochs": epochs, "seed": seed}
    history, best_epoch, sec_per_epoch = fit(model, train_loader, val_loader, criterion,
                                             optimizer, epochs, ckpt, class_names, config)
    plot_history(history, f"{PROJECT}/results/{name}_curves.png")

    model = load_model(ckpt)                           # evaluate the BEST epoch, not the last one
    metrics = evaluate_run(model, val_loader, class_names, f"{PROJECT}/results/{name}")
    if log:
        log_result(name, aug, mode, epochs, best_epoch, seed, sec_per_epoch, metrics, notes)

    if gradcam:
        gradcam_grid(model, val_ds, class_names, save_path=f"{PROJECT}/results/{name}_gradcam.png")
        wrong = misclassified_indices(model, val_ds)
        print(f"{len(wrong)} of {len(val_ds)} validation images misclassified. Heatmaps for some of them:")
        gradcam_grid(model, val_ds, class_names, indices=wrong, save_path=f"{PROJECT}/results/{name}_gradcam_wrong.png")
    return model, history, metrics

In [ ]:
# 7a. Preliminary test:
run_experiment("smoke_light_lastblock", aug="light", mode="last_block", epochs=2,
               log=False, overwrite=True, gradcam=False)
run_experiment("smoke_aggressive_full", aug="aggressive", mode="full", epochs=2,
               log=False, overwrite=True, gradcam=False)

In [ ]:
# 7b. Baseline run through the full pipeline (no augmentation, frozen backbone)
_ = run_experiment("baseline_none_frozen", aug="none", mode="frozen", epochs=10, gradcam = False,
                   notes="week 3 pipeline check; compare with the Week 2 baseline")

In [ ]:
# 7c. The results table so far
import pandas as pd
pd.read_csv(f"{PROJECT}/results/experiments.csv")

In [ ]:
   model = load_model(f"{PROJECT}/checkpoints/baseline_none_frozen.pt")
   _, val_ds = make_datasets(get_train_tfm("none"))
   gradcam_grid(model, val_ds, class_names, save_path=f"{PROJECT}/results/baseline_none_frozen_gradcam.png")

## EXPERIMENTS:

1. Baseline seed 1, with no augmentation

- Seeding helps understand how much variables like starting weights of the new last layer, the order of the photos per epoch, random augmentation choices (for aug runs) and small effects on the GPU can make two identical runs differ.

In [ ]:
run_experiment("baseline_none_frozen_seed1", aug="none", mode="frozen", epochs=10, seed=1, gradcam=False)

2. Baseline, seed 2, no augmentation

In [ ]:
run_experiment("baseline_none_frozen_seed2", aug="none", mode="frozen", epochs=10, seed=2, gradcam=False)

## Stage A: augmentation (Grad-CAM on)
1. LIGHT AUGMENTATION: flips, rotation up to 10°

In [ ]:
run_experiment("aug_light_frozen", aug="light", mode="frozen", epochs=10)

2. AGGRESSIVE AUGMENTATION: random crops (70 to 100% of the photo), flips, rotation up to 25°, and mild colour jitter

In [ ]:
run_experiment("aug_aggressive_frozen", aug="aggressive", mode="frozen", epochs=10)

## Stage B: Fine Tuning, No Augmentation

In [ ]:
BEST_AUG = "none"

1. last layer plus the last convolutional block

In [ ]:
run_experiment(f"ft_lastblock_{BEST_AUG}", aug=BEST_AUG, mode="last_block", epochs=10)

2. All layers

In [ ]:
run_experiment(f"ft_full_{BEST_AUG}", aug=BEST_AUG, mode="full", epochs=10)